In [1]:
import pandas as pd
import json
import re

In [2]:
with open('data/raw.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

df = pd.DataFrame(data)

df_docs = df[['review_id', 'place_name', 'review_text', 'rating', 'published_at_date']].copy()
df_docs.columns = ['doc_id', 'place_name', 'text', 'rating', 'date']

print(f"Завантажено записів: {len(df_docs)}")

Завантажено записів: 6174


In [ ]:
def clean_text(text):
    if not isinstance(text, str): return ""
    text = re.sub(r'\s+', ' ', text).strip()
    text = text.replace("'", "’").replace("`", "’").replace("'", "’")
    text = re.sub(r'http\S+|www\S+', '<URL>', text)
    text = re.sub(r'\S+@\S+', '<EMAIL>', text)
    text = re.sub(r'\+?\d{1,3}[-.\s]?\(?\d{2,3}\)?[-.\s]?\d{3}[-.\s]?\d{2}[-.\s]?\d{2}', '<PHONE>', text)
    return text

In [ ]:
df_docs['text'] = df_docs['text'].apply(clean_text)

df_docs['char_len'] = df_docs['text'].str.len()
df_docs['word_count'] = df_docs['text'].str.split().str.len()

df_docs = df_docs[df_docs['word_count'] > 0]

duplicates_count = df_docs.duplicated(subset=['text']).sum()
short_reviews = df_docs[df_docs['word_count'] < 5].shape[0]
junk_rows = df_docs[df_docs['text'].str.match(r'^[0-9\W]+$')].shape[0]

print("--- Статистика аудиту ---")
print(f"Загальна кількість документів: {len(df_docs)}")
print(f"Медіанна довжина (слів): {df_docs['word_count'].median()}")
print(f"Дублікати: {duplicates_count} ({duplicates_count/len(df_docs)*100:.2f}%)")
print(f"Короткі відгуки (<5 слів): {short_reviews}")
print(f"Сміттєві рядки (тільки символи/цифри): {junk_rows}")

df_docs.to_csv('data/processed.csv', index=False, encoding='utf-8-sig')

--- Статистика аудиту ---
Загальна кількість документів: 4640
Медіанна довжина (слів): 14.0
Дублікати: 68 (1.47%)
Короткі відгуки (<5 слів): 664
Сміттєві рядки (тільки символи/цифри): 13


Зібраний датасет складається з понад 6000 реальних відгуків про заклади Львова зпаршених через open-source програму - https://github.com/omkarcloud/google-maps-scraper?tab=readme-ov-file. На основі цього створив якісну базу для пошукової системи. Дані охоплюють різні типи ресторанів — від грузинської кухні до стендап-клубів, що робить пошук різноманітним. Головним ризиком є велика кількість коротких відгуків ("Смачно", "Супер"), які не несуть корисної інформації для користувача. Також у текстах зустрічається суржик та діалектизми, які можуть бути складними для стандартних алгоритмів. У наступній лабораторній роботі потрібно обов’язково провести лематизацію, щоб система розуміла зв’язок між словами у різних відмінках.